In [ ]:
import os, re, json
import numpy as np, pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt, seaborn as sns
from datetime import datetime
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_recall_fscore_support, classification_report, confusion_matrix
)

In [ ]:
base_dir = "./Aggregated"
AGGREGATED_LABELS = os.path.join(base_dir, "AllTales-Aggregated.csv")

SENTIMENT_COL = "Sentiments-Aggregated"
MULTI_FLAG_COL = "Multi-Aggregated"
JOIN_KEYS = ["Story", "Segment"]

SENTIMENTS = ["Negative", "Neutral", "Positive"]

USE_GPU = True
EARLY_STOP_ROUNDS = 150
N_ESTIMATORS_MAX = 2500
SAVE_CM_PLOTS = True
SMOOTH_WINDOW = 3
CB_BETA = 0.99
RANDOM_STATE = 42
GRID_CV_SPLITS = 4
GRIDSEARCH_N_JOBS = 1
GRIDSEARCH_VERBOSE = 2

PARAM_GRID = {
    "max_depth": [4, 5],
    "min_child_weight": [1.5],
    "learning_rate": [0.05, 0.06],
    "gamma": [0.15, 0.2, 0.5],
    "subsample": [0.85, 0.9],
    "colsample_bytree": [0.75, 0.8],
    "reg_lambda": [2.0, 2.5],
    "reg_alpha": [0.4, 0.5, 0.6],
    "scale_pos_weight": [0.9],
}

timestamp = datetime.now().strftime("%y%m%d-%H%M")
out_dir = os.path.join(base_dir, "results", f"{timestamp}-GridSearch-sentiment")
os.makedirs(out_dir, exist_ok=True)

In [ ]:
def class_balanced_weights(y, num_classes, beta=0.99):
    counts = np.bincount(y, minlength=num_classes).astype(float)
    eff_num = 1.0 - np.power(beta, counts)
    eff_num[eff_num == 0] = 1.0
    w = (1.0 - beta) / eff_num
    w *= num_classes / np.sum(w)
    return w

def build_sample_weights(y, class_w):
    return class_w[y]

def variance_filter_columns(X):
    return list(X.columns[X.var(axis=0) > 0.0])

def storywise_zscore_fit(df, cols):
    stats = {}
    for s, g in df.groupby("Story"):
        mu, sd = g[cols].mean(), g[cols].std().replace(0, 1)
        stats[s] = (mu, sd)
    return stats

def storywise_zscore_transform(df, cols, stats):
    out = df.copy()
    for s, idx in out.groupby("Story").groups.items():
        mu, sd = stats.get(s, (out[cols].mean(), out[cols].std().replace(0, 1)))
        out.loc[idx, cols] = (out.loc[idx, cols] - mu) / sd
    return out

def smooth_probs_per_story(df, y_proba, class_names, window=3):
    if y_proba.size == 0:
        return y_proba
    df_local = df.reset_index(drop=True)
    proba_df = pd.DataFrame(y_proba, columns=class_names, index=df_local.index)
    proba_df = proba_df.join(df_local[["Story"]])
    smoothed = []
    for _, g in proba_df.groupby("Story", sort=False):
        w = max(1, min(window, max(1, len(g) // 6)))
        if len(g) < 20:
            roll = g[class_names]
        else:
            roll = g[class_names].rolling(w, center=True, min_periods=1).median()
        roll.index = g.index
        smoothed.append(roll)
    return pd.concat(smoothed).sort_index().values

class StoryWiseXGBClassifier(BaseEstimator, ClassifierMixin):
    def __init__(
        self,
        num_class,
        use_gpu=True,
        cb_beta=0.99,
        smooth_window=3,
        random_state=42,
        n_estimators=2500,
        learning_rate=0.05,
        max_depth=8,
        min_child_weight=5,
        subsample=0.9,
        colsample_bytree=0.9,
        gamma=0.0,
        reg_lambda=1.0,
        reg_alpha=0.0,
        scale_pos_weight=1.0,
        n_jobs=-1,
        importance_type="gain",
        verbosity=0,
    ):
        self.num_class = num_class
        self.use_gpu = use_gpu
        self.cb_beta = cb_beta
        self.smooth_window = smooth_window
        self.random_state = random_state
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.min_child_weight = min_child_weight
        self.subsample = subsample
        self.colsample_bytree = colsample_bytree
        self.gamma = gamma
        self.reg_lambda = reg_lambda
        self.reg_alpha = reg_alpha
        self.scale_pos_weight = scale_pos_weight
        self.n_jobs = n_jobs
        self.importance_type = importance_type
        self.verbosity = verbosity

    def _build_estimator(self):
        params = dict(
            n_estimators=self.n_estimators,
            n_jobs=self.n_jobs,
            objective="multi:softprob",
            num_class=self.num_class,
            learning_rate=self.learning_rate,
            max_depth=self.max_depth,
            min_child_weight=self.min_child_weight,
            subsample=self.subsample,
            colsample_bytree=self.colsample_bytree,
            gamma=self.gamma,
            reg_lambda=self.reg_lambda,
            reg_alpha=self.reg_alpha,
            scale_pos_weight=self.scale_pos_weight,
            random_state=self.random_state,
            importance_type=self.importance_type,
            eval_metric=["mlogloss", "merror"],
            verbosity=self.verbosity,
        )
        if self.use_gpu:
            params.update({"tree_method": "hist", "device": "cuda"})
        else:
            params.update({"tree_method": "hist", "device": "cpu"})
        return xgb.XGBClassifier(**params)

    def _normalize(self, X):
        X_local = X.copy()
        self.feature_cols_ = [c for c in X_local.columns if c != "Story"]
        if "Story" in X_local.columns:
            X_local = storywise_zscore_transform(
                X_local,
                self.feature_cols_,
                storywise_zscore_fit(X_local, self.feature_cols_),
            )
        return X_local

    def fit(self, X, y):
        X_local = self._normalize(X)
        self.kept_cols_ = variance_filter_columns(X_local[self.feature_cols_])
        if not self.kept_cols_:
            self.kept_cols_ = list(self.feature_cols_)

        class_w = class_balanced_weights(np.asarray(y), self.num_class, beta=self.cb_beta)
        sample_weight = build_sample_weights(np.asarray(y), class_w)

        self.model_ = self._build_estimator()
        self.model_.fit(X_local[self.kept_cols_], np.asarray(y), sample_weight=sample_weight, verbose=False)
        self.classes_ = np.arange(self.num_class)
        return self

    def predict_proba(self, X):
        X_local = self._normalize(X)
        y_proba = self.model_.predict_proba(X_local[self.kept_cols_])
        if "Story" in X_local.columns:
            y_proba = smooth_probs_per_story(
                X_local[["Story"]],
                y_proba,
                SENTIMENTS,
                self.smooth_window,
            )
        return y_proba

    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)

    @property
    def feature_importances_(self):
        return self.model_.feature_importances_


def macro_f1_grid_scorer(estimator, X, y):
    y_pred = estimator.predict(X)
    _, _, f1_macro, _ = precision_recall_fscore_support(
        y,
        y_pred,
        average="macro",
        zero_division=0,
    )
    return f1_macro

Load

In [ ]:
if not os.path.exists(AGGREGATED_LABELS):
    raise SystemExit(f"Label file not found: {AGGREGATED_LABELS}")

labels_df = pd.read_csv(AGGREGATED_LABELS)
labels_df = labels_df.dropna(subset=[SENTIMENT_COL])

labels_df[SENTIMENT_COL] = labels_df[SENTIMENT_COL].astype(str).str.strip().str.title()
labels_df[MULTI_FLAG_COL] = labels_df[MULTI_FLAG_COL].astype(str).str.strip().str.lower()
labels_df = labels_df[labels_df[SENTIMENT_COL].isin(SENTIMENTS)].copy()
labels_df = labels_df[labels_df[MULTI_FLAG_COL] != "yes"].copy()

labels_df["Story"] = (
    labels_df["Story"].astype(str)
    .apply(lambda s: re.sub(r"^\s*\d+\s*-\s*", "", s.strip()))
    .str.strip().str.title()
)

print(f"Loaded {len(labels_df)} total rows after filtering.")

Merge Labels + Features

In [ ]:
labels_df["Story"] = labels_df["Story"].astype(str).apply(
    lambda s: re.sub(r"^\s*\d+-\s*", "", s.strip())
).str.lower()
labels_df["Segment"] = labels_df["Segment"].astype(str).str.strip()

feature_folders = {
    re.sub(r"^\s*\d+-\s*", "", f.strip()).lower(): f
    for f in os.listdir(base_dir)
    if os.path.isdir(os.path.join(base_dir, f))
}

merged = []
for story, df_lab in labels_df.groupby("Story"):
    folder = feature_folders.get(story)
    if not folder:
        print(f"Missing folder: {story}")
        continue

    feat_path = os.path.join(base_dir, folder, "AllFront_features.csv")
    if not os.path.exists(feat_path):
        print(f"Missing features: {story}")
        continue

    df_feat = pd.read_csv(feat_path)
    df_feat["Story"] = df_feat["Story"].astype(str).str.lower()
    df_feat["Segment"] = df_feat["Segment"].astype(str).str.strip()

    df_merged = pd.merge(df_lab, df_feat, on=["Story", "Segment"], how="inner")
    if df_merged.empty:
        print(f"No overlap: {story}")
        continue

    merged.append(df_merged)
    # print(f"{Story}: {len(df_merged)} rows merged")

if not merged:
    raise SystemExit("No matching stories found.")

df_all = pd.concat(merged, ignore_index=True)
print(f"Total merged rows: {len(df_all)}")

# --- Quick missing check ---
# missing = set(zip(labels_df["Story"], labels_df["Segment"])) - set(zip(df_all["Story"], df_all["Segment"]))
# print(f"{len(missing)} labeled segments had no matching features.")

Data and Split

In [ ]:
drop_cols = [
    "Story", "Segment", SENTIMENT_COL, "text_original", "Multi", "Sentiments-Perplexity", "Multi-Perplexity",
    "Sentiments-GPT5", "Multi-GPT5", "Sentiments-Mistral", "Multi-Mistral",
    "Sentiments-GPTOSS20B", "Multi-GPTOSS20B", "Sentiment", "Multi-Aggregated",
    "torso_pitch__mean", "torso_pitch__std", "torso_roll__mean", "torso_roll__std", "torso_yaw__mean", "torso_yaw__std"
]
feat_cols = [c for c in df_all.columns if c not in drop_cols and np.issubdtype(df_all[c].dtype, np.number)]
features_keep = ["mouthSmileRight_mean", "mouthSmileLeft_mean", "pose_RIGHT_ELBOW_y_mean", "browDownRight_mean", "mouthFrownLeft_std", "mouthSmileRight_std", "pose_LEFT_HIP_y_velocity_std", "mouthRollLower_mean", "browDownLeft_mean", "mouthUpperUpRight_mean", "mouthShrugLower_std", "browDownLeft_std", "right_hand_WRIST_y_velocity_std", "dist_left_wrist_to_left_shoulder_peaks_per_s", "pose_RIGHT_HIP_y_acceleration_std", "left_hand_WRIST_x_peaks_per_s", "pose_LEFT_SHOULDER_z_acceleration_std", "mouthRollUpper_std", "left_hand_WRIST_y_mean", "head_pitch_deg_mean", "pose_LEFT_SHOULDER_y_velocity_std", "pose_LEFT_HIP_y_mean", "mouthClose_mean", "jawRight_mean", "left_hand_WRIST_y_std", "browOuterUpRight_std", "pose_LEFT_SHOULDER_y_acceleration_std", "eyeLookOutRight_mean", "browDownRight_std", "mouthPucker_mean", "pose_LEFT_SHOULDER_x_acceleration_std", "mouthRight_mean", "pose_LEFT_SHOULDER_z_std", "mouthPressRight_mean", "right_hand_WRIST_x_acceleration_mean", "mouthRollLower_std", "right_hand_WRIST_y_mean", "mouthSmileLeft_std", "mouthDimpleRight_std", "pose_LEFT_ELBOW_y_std", "pose_LEFT_HIP_z_peaks_per_s", "right_arm_angle_std", "eyeWideLeft_mean", "jawOpen_std", "mouthFrownRight_peaks_per_s", "eyeWideLeft_std", "dist_right_wrist_to_nose_peaks_per_s", "mouthUpperUpRight_std", "mouthDimpleLeft_std", "pose_LEFT_ELBOW_y_mean", "eyeLookInRight_peaks_per_s", "R_SHOULDER_accum_dist_avg", "right_hand_WRIST_y_acceleration_std", "pose_LEFT_HIP_y_std", "right_hand_WRIST_x_velocity_mean", "jawRight_std", "right_hand_WRIST_z_acceleration_mean", "noseSneerRight_std", "mouthRight_std", "pose_NOSE_z_peaks_per_s", "pose_LEFT_HIP_y_acceleration_std", "mouthShrugLower_mean", "pose_RIGHT_SHOULDER_y_acceleration_mean", "eyeSquintLeft_mean", "mouthFunnel_mean", "pose_NOSE_y_mean", "pose_LEFT_SHOULDER_z_peaks_per_s", "mouthLowerDownLeft_std", "mouthDimpleRight_mean", "eyeBlinkLeft_peaks_per_s", "right_hand_WRIST_z_mean", "mouthPressLeft_std", "dist_right_wrist_to_nose_avg", "eyeLookUpLeft_mean", "dist_left_wrist_to_nose_avg", "eyeLookUpRight_std", "pose_LEFT_HIP_z_velocity_mean", "mouthUpperUpLeft_mean", "pose_LEFT_ELBOW_z_velocity_mean", "browOuterUpLeft_std", "pose_RIGHT_HIP_x_acceleration_std", "pose_NOSE_z_std", "NOSE_accum_dist_avg", "eyeSquintRight_mean", "pose_RIGHT_HIP_x_peaks_per_s", "mouthLowerDownLeft_mean", "pose_RIGHT_WRIST_y_peaks_per_s", "left_hand_WRIST_y_acceleration_mean", "pose_LEFT_ELBOW_z_mean", "pose_LEFT_SHOULDER_x_acceleration_mean", "pose_RIGHT_HIP_y_velocity_mean", "pose_LEFT_ELBOW_x_peaks_per_s", "pose_RIGHT_ELBOW_y_acceleration_std", "dist_elbows_lr_avg", "mouthStretchLeft_mean", "eyeLookDownRight_mean", "pose_RIGHT_SHOULDER_x_acceleration_mean", "pose_LEFT_HIP_x_acceleration_mean", "head_yaw_deg_mean", "head_roll_deg_mean", "eyeBlinkLeft_std", "pose_RIGHT_SHOULDER_z_peaks_per_s", "pose_RIGHT_ELBOW_y_std", "pose_LEFT_WRIST_y_peaks_per_s", "pose_RIGHT_WRIST_x_peaks_per_s", "pose_LEFT_HIP_x_peaks_per_s", "noseSneerRight_mean", "browOuterUpLeft_mean", "mouthFrownLeft_mean", "cheekPuff_peaks_per_s", "head_yaw_deg_peaks_per_s", "mouthSmileLeft_peaks_per_s", "dist_right_wrist_to_left_shoulder_avg", "mouthDimpleLeft_mean", "mouthShrugUpper_mean", "pose_RIGHT_ELBOW_z_acceleration_mean", "mouthStretchRight_mean", "pose_NOSE_y_peaks_per_s", "left_hand_WRIST_x_std", "eyeLookUpRight_mean", "eyeBlinkRight_mean", "eyeLookDownRight_std", "right_hand_WRIST_z_std", "browOuterUpRight_mean", "mouthFunnel_peaks_per_s", "right_hand_WRIST_x_mean", "pose_NOSE_x_acceleration_std", "mouthPressLeft_peaks_per_s"]
feat_cols = [c for c in feat_cols if c in set(features_keep)]

label_to_id = {lbl: i for i, lbl in enumerate(SENTIMENTS)}
y_all = df_all[SENTIMENT_COL].map(label_to_id).values
groups_all = df_all["Story"].values
X_search = df_all[["Story"] + feat_cols].copy()

NUM_CLASS = len(SENTIMENTS)
print(feat_cols)
print(f"[GridSearchCV] CV={GRID_CV_SPLITS} | Samples={len(X_search)} | Features={len(feat_cols)} | Groups={len(np.unique(groups_all))}")


Training

In [ ]:
# Grid search for hyperparameter selection

cv_splitter = StratifiedGroupKFold(
    n_splits=GRID_CV_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)
cv_splits = list(cv_splitter.split(X_search, y_all, groups_all))

fold_rows = []
for fold_id, (_, va_idx) in enumerate(cv_splits, start=1):
    fold_tmp = df_all.iloc[va_idx][["Story", "Segment", SENTIMENT_COL]].copy()
    fold_tmp["Fold"] = fold_id
    fold_rows.append(fold_tmp)

fold_assignments = pd.concat(fold_rows, ignore_index=True)
fold_assignments.to_csv(os.path.join(out_dir, f"{timestamp}-GridSearch_fold_assignments.csv"), index=False)

base_estimator = StoryWiseXGBClassifier(
    num_class=NUM_CLASS,
    use_gpu=USE_GPU,
    cb_beta=CB_BETA,
    smooth_window=SMOOTH_WINDOW,
    random_state=RANDOM_STATE,
    n_estimators=N_ESTIMATORS_MAX,
    n_jobs=-1,
    verbosity=0,
)

grid = GridSearchCV(
    estimator=base_estimator,
    param_grid=PARAM_GRID,
    scoring=macro_f1_grid_scorer,
    n_jobs=GRIDSEARCH_N_JOBS,
    refit=True,
    cv=cv_splits,
    verbose=GRIDSEARCH_VERBOSE,
    pre_dispatch="2*n_jobs",
    error_score=np.nan,
    return_train_score=True,
)

print("Starting GridSearchCV ...")
grid.fit(X_search, y_all, groups=groups_all)
print("GridSearchCV finished.")

Save & Visualize Results

In [ ]:
best_params = grid.best_params_

fold_predictions = []

for fold_id, (tr_idx, va_idx) in enumerate(cv_splits, start=1):

    print(f"Running fold {fold_id}")

    X_tr = X_search.iloc[tr_idx]
    y_tr = y_all[tr_idx]

    X_va = X_search.iloc[va_idx]
    y_va = y_all[va_idx]

    model = StoryWiseXGBClassifier(
        num_class=NUM_CLASS,
        use_gpu=USE_GPU,
        cb_beta=CB_BETA,
        smooth_window=SMOOTH_WINDOW,
        random_state=RANDOM_STATE,
        n_estimators=N_ESTIMATORS_MAX,
        **best_params
    )

    model.fit(X_tr, y_tr)

    proba = model.predict_proba(X_va)
    pred = np.argmax(proba, axis=1)

    df_pred = df_all.iloc[va_idx][["Story","Segment",SENTIMENT_COL]].copy()

    df_pred["Fold"] = fold_id
    df_pred["y_true"] = y_va
    df_pred["y_pred"] = pred

    for i, cls in enumerate(SENTIMENTS):
        df_pred[f"proba_{cls}"] = proba[:, i]

    fold_predictions.append(df_pred)

fold_predictions_df = pd.concat(fold_predictions, ignore_index=True)

fold_predictions_df.to_csv(
    os.path.join(out_dir, f"{timestamp}-GridSearch_fold_predictions.csv"),
    index=False
)

print("Saved fold predictions.")

In [ ]:
# Save GridSearchCV outputs

cv_results_df = pd.DataFrame(grid.cv_results_).sort_values(
    by=["rank_test_score", "mean_test_score"],
    ascending=[True, False],
).reset_index(drop=True)
cv_results_path = os.path.join(out_dir, f"{timestamp}-GridSearch_cv_results.csv")
cv_results_df.to_csv(cv_results_path, index=False)

best_params_payload = {
    "best_index": int(grid.best_index_),
    "best_score": float(grid.best_score_),
    "best_params": grid.best_params_,
    "n_splits": int(grid.n_splits_),
    "scoring": "macro_f1_grid_scorer",
}
with open(os.path.join(out_dir, f"{timestamp}-GridSearch_best_params.json"), "w") as f:
    json.dump(best_params_payload, f, indent=2)

best_summary_df = cv_results_df.loc[:, [
    "rank_test_score", "mean_test_score", "std_test_score",
    "mean_train_score", "std_train_score", "params"
]].head(20)
best_summary_df.to_csv(os.path.join(out_dir, f"{timestamp}-GridSearch_top20.csv"), index=False)

print("Best score:", grid.best_score_)
print("Best params:", grid.best_params_)
display(best_summary_df)

Aggregate Feature Importances

In [ ]:
# Save refit model diagnostics and feature importances

best_estimator = grid.best_estimator_
y_pred_full = best_estimator.predict(X_search)
y_proba_full = best_estimator.predict_proba(X_search)

acc = accuracy_score(y_all, y_pred_full)
bal_acc = balanced_accuracy_score(y_all, y_pred_full)
p_m, r_m, f1_m, _ = precision_recall_fscore_support(y_all, y_pred_full, average="macro", zero_division=0)
p_w, r_w, f1_w, _ = precision_recall_fscore_support(y_all, y_pred_full, average="weighted", zero_division=0)

final_metrics_df = pd.DataFrame([
    {
        "Accuracy": acc,
        "Balanced_Accuracy": bal_acc,
        "Macro_Precision": p_m,
        "Macro_Recall": r_m,
        "Macro_F1": f1_m,
        "Weighted_Precision": p_w,
        "Weighted_Recall": r_w,
        "Weighted_F1": f1_w,
    }
])
final_metrics_df.to_csv(os.path.join(out_dir, f"{timestamp}-GridSearch_refit_metrics.csv"), index=False)

feat_imp = pd.Series(
    best_estimator.feature_importances_,
    index=best_estimator.kept_cols_,
    name="Importance",
).sort_values(ascending=False)
feat_imp.to_csv(os.path.join(out_dir, f"{timestamp}-GridSearch_best_feature_importances.csv"))

plt.figure(figsize=(6, 4))
sns.barplot(
    x=feat_imp.head(20).values,
    y=feat_imp.head(20).index,
    palette="viridis",
)
plt.title("Top 20 Feature Importances from Best GridSearchCV Model")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig(os.path.join(out_dir, f"{timestamp}-GridSearch_top20_features.png"), dpi=300)
plt.close()

score_cols = [c for c in cv_results_df.columns if c.startswith("split") and c.endswith("_test_score")]
if score_cols:
    plt.figure(figsize=(10, 5))
    for col in score_cols:
        plt.plot(cv_results_df.index + 1, cv_results_df[col], marker="o", label=col)
    plt.plot(cv_results_df.index + 1, cv_results_df["mean_test_score"], marker="o", linewidth=2, label="mean_test_score")
    plt.xlabel("Parameter setting rank order")
    plt.ylabel("Macro F1")
    plt.title("GridSearchCV split-wise test scores")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{timestamp}-GridSearch_split_scores.png"), dpi=300)
    plt.close()

print("\nFinished. Results saved in:", out_dir)